### Links and Resources
| Concept / Command | Description |
|---|---|
| [Stream-static joins](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#stream-static-joins) | Joining a stream with a static DataFrame |
| [Stream-stream joins](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#stream-stream-joins) | Joining two streams — requires watermarks |
| [Supported join types](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#support-matrix-for-joins-in-streaming-queries) | Which join types are allowed in each mode |
| [`expr()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.expr.html) | SQL expression syntax for time-range join conditions |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel — 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()


def display(df, n=20):
    df.show(n, truncate=False)

# Streaming Joins

Structured Streaming supports two fundamentally different kinds of join:

| Join type | Left side | Right side | State needed | Watermark required |
|---|---|---|---|---|
| **Stream-static** | Stream | `spark.read` DataFrame | No | No |
| **Stream-stream** | Stream | Stream | Yes — both sides buffered | **Yes** — to bound state |

Stream-static joins are the common case: enrich every incoming event with data from a lookup table. The static side is loaded once per micro-batch (Spark re-evaluates it, picking up any updates).

Stream-stream joins require watermarks on **both** sides and a time-range condition in the join predicate — without it, Spark must buffer every row from both streams forever.

## Setup — streams and static lookup table

In [ ]:
import time
from pyspark.sql import functions as F

# ── static lookup: user profiles in Delta ─────────────────────────────────────
try:
    spark.sql("SELECT 1 FROM population_metrics.default.streaming_users LIMIT 1")
    print("streaming_users already exists.")
except Exception:
    from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
    users_data = [
        (0, "alice", "premium", 0.95),
        (1, "bob",   "free",    0.42),
        (2, "carol", "premium", 0.88),
        (3, "dave",  "free",    0.31),
        (4, "eve",   "premium", 0.76),
    ]
    users_schema = StructType([
        StructField("user_id",    IntegerType()),
        StructField("username",   StringType()),
        StructField("tier",       StringType()),
        StructField("risk_score", DoubleType()),
    ])
    (spark.createDataFrame(users_data, schema=users_schema)
         .write.format("delta").mode("overwrite")
         .saveAsTable("population_metrics.default.streaming_users"))
    print("streaming_users created.")

# ── stream A: orders ──────────────────────────────────────────────────────────
orders = (spark.readStream
    .format("rate").option("rowsPerSecond", 5).load()
    .withColumnRenamed("timestamp", "order_time")
    .withColumnRenamed("value",     "order_id")
    .withColumn("user_id", (F.col("order_id") % 5).cast("int"))
    .withColumn("amount",  (F.rand() * 200 + 10).cast("int")))

# ── stream B: payments ────────────────────────────────────────────────────────
payments = (spark.readStream
    .format("rate").option("rowsPerSecond", 4).load()
    .withColumnRenamed("timestamp", "payment_time")
    .withColumnRenamed("value",     "payment_id")
    .withColumn("user_id",     (F.col("payment_id") % 5).cast("int"))
    .withColumn("payment_amt", (F.rand() * 200 + 10).cast("int")))

print("orders.isStreaming:", orders.isStreaming)
print("payments.isStreaming:", payments.isStreaming)

In [ ]:
def peek(stream_df, name, seconds=8, limit=20, output_mode="append", order_by=None):
    q = (stream_df.writeStream
         .format("memory")
         .queryName(name)
         .outputMode(output_mode)
         .start())
    time.sleep(seconds)
    q.stop()
    order_clause = f"ORDER BY {order_by}" if order_by else ""
    return spark.sql(f"SELECT * FROM {name} {order_clause} LIMIT {limit}")

## 1. Stream-static inner join

Enrich every order with the user's profile from the static Delta table.

**Key properties:**
- The static DataFrame is re-evaluated each micro-batch — picks up any table updates automatically.
- No watermark needed — state is only maintained for the (finite) static side.
- All standard join types work: inner, left outer, right outer, cross.

In [ ]:
users_static = spark.read.table("population_metrics.default.streaming_users")

enriched = orders.join(users_static, on="user_id", how="inner")

display(peek(enriched, "stream_static_inner", order_by="order_time DESC"))

## 2. Stream-static left outer join

Use a left outer join when some stream records may have no matching profile — those rows pass through with `NULL` values instead of being dropped.

In [ ]:
# Inject a "ghost" user_id (99) that has no entry in the lookup table
orders_with_ghost = orders.withColumn(
    "user_id",
    F.when(F.col("order_id") % 20 == 0, F.lit(99)).otherwise(F.col("user_id"))
)

enriched_left = orders_with_ghost.join(users_static, on="user_id", how="left")

result = peek(enriched_left, "stream_static_left", seconds=8, order_by="order_time DESC")
print("Rows where user_id=99 have NULL username/tier (no profile match):")
display(result)

## 3. Stream-stream join requirements

Joining two streams is more complex because **both sides are unbounded**.
Spark must buffer rows from both sides waiting for a match to arrive on the other side.

Without bounds, those buffers grow forever. Two things are required:

### Watermarks on both sides
```python
stream_a.withWatermark("ts_a", "20 seconds")
stream_b.withWatermark("ts_b", "20 seconds")
```

### A time-range condition in the join predicate
```python
.join(..., expr("""
    a.user_id = b.user_id AND
    a.order_time BETWEEN b.payment_time - INTERVAL 30 SECONDS
                     AND b.payment_time + INTERVAL 30 SECONDS
"""))
```

Together these tell Spark: "an order from 12:00:00 can only match a payment within ±30 s."
Once the watermark advances past `order_time + 30 s`, that order can be evicted from the buffer.

## 4. Stream-stream inner join

In [ ]:
from pyspark.sql.functions import expr

orders_wm   = orders.withWatermark("order_time",    "20 seconds")
payments_wm = payments.withWatermark("payment_time", "20 seconds")

joined = (orders_wm.alias("o").join(
    payments_wm.alias("p"),
    expr('''
        o.user_id = p.user_id
        AND o.order_time BETWEEN p.payment_time - INTERVAL 15 SECONDS
                             AND p.payment_time + INTERVAL 15 SECONDS
    '''),
    how="inner"
).select(
    F.col("o.order_id"),   F.col("o.order_time"),
    F.col("p.payment_id"), F.col("p.payment_time"),
    F.col("o.user_id"),
    F.col("o.amount"),     F.col("p.payment_amt")
))

result = peek(joined, "ss_inner", seconds=12, output_mode="append",
              order_by="order_time DESC")
print("Matched order-payment pairs (joined within ±15 s on user_id):")
display(result)

## 5. Stream-stream left outer join

Left outer joins between two streams **require watermarks on both sides** and emit unmatched left rows only after the watermark ensures no right-side match can still arrive.

Unmatched rows appear with `NULL` values for the right-side columns — identical to batch left outer join behaviour.

In [ ]:
left_joined = (orders_wm.alias("o").join(
    payments_wm.alias("p"),
    expr('''
        o.user_id = p.user_id
        AND o.order_time BETWEEN p.payment_time - INTERVAL 10 SECONDS
                             AND p.payment_time + INTERVAL 10 SECONDS
    '''),
    how="left_outer"
).select(
    F.col("o.order_id"),   F.col("o.order_time"),
    F.col("o.user_id"),    F.col("o.amount"),
    F.col("p.payment_id"), F.col("p.payment_time"), F.col("p.payment_amt")
))

result = peek(left_joined, "ss_left", seconds=15, output_mode="append",
              order_by="order_time DESC")
print("All orders present; payment columns NULL when no payment matched within the time window:")
display(result)

## 6. State management in stream-stream joins

For a stream-stream join, Spark maintains two buffers in the state store:

```
  ┌──────────────────────────────────────┐
  │ Left buffer  (orders not yet matched) │  ← grows until a right-side match arrives
  │                                      │     or watermark evicts the row
  └──────────────────────────────────────┘
  ┌──────────────────────────────────────┐
  │ Right buffer (payments not matched)  │  ← same lifecycle
  └──────────────────────────────────────┘
```

**State is evicted when:**
`left_event_time + right_watermark_delay + time_range_upper_bound < current_watermark`

In practice: set watermarks to the maximum expected delivery delay plus the time-range window.
Undersized watermarks drop valid matches. Oversized watermarks waste state store memory.

### Supported join types in streaming

| Join type | Stream-static | Stream-stream |
|---|---|---|
| Inner | ✅ | ✅ (watermark + time range required) |
| Left outer | ✅ | ✅ (both watermarks required) |
| Right outer | ✅ | ✅ (both watermarks required) |
| Full outer | ✅ | ❌ not supported |
| Left semi | ✅ | ✅ |
| Left anti | ✅ | ❌ not supported |

In [ ]:
# Inspect state store metrics from a running stream-stream query
q = (joined.writeStream
     .format("memory").queryName("state_inspect")
     .outputMode("append").start())

time.sleep(10)

prog = q.lastProgress
if prog and prog.get("stateOperators"):
    for i, op in enumerate(prog["stateOperators"]):
        print(f"State operator {i}:")
        print(f"  numRowsTotal:    {op.get('numRowsTotal', 'n/a')}")
        print(f"  memoryUsedBytes: {op.get('memoryUsedBytes', 'n/a')}")
        print(f"  numRowsDroppedByWatermark: {op.get('numRowsDroppedByWatermark', 'n/a')}")
else:
    print("No stateOperators in lastProgress — run longer for metrics to appear.")

q.stop()

## Takeaways

- **Stream-static joins** are the workhorse: enrich every event with data from a lookup table. No watermark needed; all join types work; the static side refreshes each batch.
- **Stream-stream joins** require **watermarks on both sides** and a **time-range condition** to bound how long unmatched rows are buffered.
- The time-range condition (`ts_a BETWEEN ts_b - X AND ts_b + Y`) directly controls state size — tighter ranges mean less memory.
- For inner and outer stream-stream joins use `outputMode("append")`.
- Unmatched rows in a left outer stream-stream join are emitted only once the watermark guarantees no further match can arrive.
- Full outer and left anti joins are not supported between two streams.
- Monitor state operator metrics (`query.lastProgress["stateOperators"]`) in production to catch unexpected state growth early.